# Which Berke lab sessions need sorting / decoding?

Finds every Berke lab hex maze session and works out what the **full sorting + decode
pipeline** (`Berke_Lab_Sorting_and_Decode_V1.ipynb`) still needs to be run on -- the same
detect-what-can-be-populated idea as `Hex_Maze_Theta.ipynb` and `Populate_All_Decoding.ipynb`.

That pipeline is: sort groups -> SpikeSortingRecording -> ArtifactDetection -> SpikeSorting
-> CurationV1 -> MetricCuration -> SpikeSortingOutput -> SortedSpikesGroup + PositionGroup
-> SortedSpikesDecodingV1 -> DecodingOutput.

So a session is *runnable* when it has:
- **raw ephys** (`Raw`) -- otherwise there is nothing to sort
- **sort groups** (`spikesorting.v1.SortGroup`) -- the pipeline sorts per sort group
- **Processed position** (`PositionOutput.TrodesPosV1`) -- what `PositionGroup` is built from,
  needed for the decode half

This notebook only reads -- it doesn't populate anything.

## Find the Berke lab hex maze sessions

In [1]:
import pandas as pd

import spyglass.common as sgc
import spyglass.spikesorting.v1 as sgs
from spyglass.common import Raw
from spyglass.position import PositionOutput
from spyglass.decoding.decoding_merge import DecodingOutput
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
from spyglass_hexmaze.hex_maze_behavior import HexMazeBlock

# All hex maze sessions, with their lab + subject
hex_sessions = sorted(set(HexMazeBlock.fetch('nwb_file_name')))
session_keys = [{'nwb_file_name': s} for s in hex_sessions]
session_lab = pd.DataFrame(
    (sgc.Session & session_keys).fetch('nwb_file_name', 'subject_id', 'lab_name', as_dict=True)
)

# Just the Berke lab ones (these are the IM-* sessions, sorted with spikesorting v1)
berke = session_lab[session_lab['lab_name'] == 'Berke Lab'].copy()
berke_sessions = sorted(berke['nwb_file_name'])

print(f"{len(berke_sessions)} Berke lab hex maze sessions "
      f"across {berke['subject_id'].nunique()} subjects")

/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-07-27 20:41:08,040][INFO]: DataJoint 0.14.6 connected to scrater@lmf-db.cin.ucsf.edu:3306
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-fiber-photometry': 
 ndx-fiber-photometry defines OpticalFiber.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines ExcitationSource.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines Photodetector.model as an attribute (dtype: text) while the core 

51 Berke lab hex maze sessions across 9 subjects


## Check each pipeline prerequisite

Each of these is session-level for Berke (one run epoch per session, `00_r1`).

In [ ]:
berke_keys = [{'nwb_file_name': s} for s in berke_sessions]

# Raw ephys: nothing to sort without it
ephys_sessions = set((Raw & berke_keys).fetch('nwb_file_name'))

# Sort groups: Berke IM-* sessions live in spikesorting v1
# (the pipeline sorts per sort group, so these must exist first -- set_group_by_shank)
sort_group_sessions = {
    s for s in berke_sessions if len(sgs.SortGroup & {'nwb_file_name': s}) > 0
}

# Processed position: what PositionGroup is built from for the decode half
processed_position = pd.DataFrame(
    PositionOutput.TrodesPosV1.fetch('nwb_file_name', 'interval_list_name', as_dict=True)
)
pos_sessions = set(processed_position['nwb_file_name']) & set(berke_sessions) if len(processed_position) else set()

# Already sorted? (walk back through the v1 recording selection)
# NOTE: don't use merge_fetch('nwb_file_name') -- the v1 part is keyed by sorting_id, so it
# would be skipped and every v1-sorted session would look unsorted.
sorted_sessions = {
    s for s in berke_sessions
    if len(SpikeSortingOutput().get_restricted_merge_ids(
        {'nwb_file_name': s}, sources=['v0', 'v1'], restrict_by_artifact=False))
}

# Already decoded? (walk the DecodingOutput parts)
dec_frames = []
for part in DecodingOutput().parts(as_objects=True):
    if 'nwb_file_name' in part.heading.names:
        dec_frames.append(pd.DataFrame(part.fetch('nwb_file_name', as_dict=True)))
dec = pd.concat(dec_frames, ignore_index=True) if dec_frames else pd.DataFrame(columns=['nwb_file_name'])
decoded_sessions = set(dec['nwb_file_name']) & set(berke_sessions)

print(f"raw ephys:      {len(ephys_sessions)} / {len(berke_sessions)}")
print(f"sort groups:    {len(sort_group_sessions)} / {len(berke_sessions)}")
print(f"processed position:{len(pos_sessions)} / {len(berke_sessions)}")
print(f"sorted:         {len(sorted_sessions)} / {len(berke_sessions)}")
print(f"decoded:        {len(decoded_sessions)} / {len(berke_sessions)}")

raw ephys:      27 / 51
sort groups:    27 / 51
processed position:31 / 51
sorted:         26 / 51
decoded:        26 / 51


## What does each session need next?

The pipeline is sequential, so we report the **first** missing step for each session.

In [4]:
berke['has_ephys'] = berke['nwb_file_name'].isin(ephys_sessions)
berke['has_sort_group'] = berke['nwb_file_name'].isin(sort_group_sessions)
berke['has_position'] = berke['nwb_file_name'].isin(pos_sessions)
berke['has_sorting'] = berke['nwb_file_name'].isin(sorted_sessions)
berke['has_decode'] = berke['nwb_file_name'].isin(decoded_sessions)


def next_step(row):
    """First missing step in the sorting -> decode pipeline for this session."""
    if not row.has_ephys:
        return 'no ephys (cannot sort)'
    if not row.has_sort_group:
        return '1. create sort groups'
    if not row.has_sorting:
        return '2. run sorting'
    if not row.has_position:
        return '3. process position (needed to decode)'
    if not row.has_decode:
        return '4. run decode'
    return 'done'


berke['next_step'] = berke.apply(next_step, axis=1)

candidates = (
    berke[['subject_id', 'nwb_file_name', 'has_ephys', 'has_sort_group',
           'has_position', 'has_sorting', 'has_decode', 'next_step']]
    .sort_values(['next_step', 'subject_id', 'nwb_file_name'])
    .reset_index(drop=True)
)

print('Berke sessions by next step:')
print(candidates['next_step'].value_counts().to_string())
display(candidates)

Berke sessions by next step:
next_step
done                      26
no ephys (cannot sort)    24
2. run sorting             1


,subject_id,nwb_file_name,has_ephys,has_sort_group,has_position,has_sorting,has_decode,next_step
0,IM-1947,IM-1947_20260404_.nwb,True,True,True,False,False,2. run sorting
1,IM-1478,IM-1478_20220719_.nwb,True,True,True,True,True,done
2,IM-1478,IM-1478_20220720_.nwb,True,True,True,True,True,done
3,IM-1478,IM-1478_20220724_.nwb,True,True,True,True,True,done
4,IM-1478,IM-1478_20220725_.nwb,True,True,True,True,True,done
5,IM-1478,IM-1478_20220726_.nwb,True,True,True,True,True,done
6,IM-1478,IM-1478_20220727_.nwb,True,True,True,True,True,done
7,IM-1594,IM-1594_20230725_.nwb,True,True,True,True,True,done
8,IM-1594,IM-1594_20230726_.nwb,True,True,True,True,True,done
9,IM-1594,IM-1594_20230727_.nwb,True,True,True,True,True,done


## The run list

Sessions ready for each stage, as `nwb_file_name`s you can paste straight into
`Berke_Lab_Sorting_and_Decode_V1.ipynb`.

In [5]:
# Sessions that have everything the sorting pipeline needs and just haven't been run yet
ready_to_sort = candidates.loc[candidates['next_step'] == '2. run sorting', 'nwb_file_name'].tolist()
# Sorted already, position ready, just needs the decode half
ready_to_decode = candidates.loc[candidates['next_step'] == '4. run decode', 'nwb_file_name'].tolist()
# Have ephys but no sort groups -- do this first (set_group_by_shank), then they can be sorted
need_sort_groups = candidates.loc[candidates['next_step'] == '1. create sort groups', 'nwb_file_name'].tolist()
# Sorted but missing position, so the decode half is blocked
need_position = candidates.loc[candidates['next_step'] == '3. process position (needed to decode)', 'nwb_file_name'].tolist()

for label, sessions in [
    ('READY TO SORT (run the full pipeline)', ready_to_sort),
    ('READY TO DECODE (sorted already)', ready_to_decode),
    ('NEED SORT GROUPS FIRST', need_sort_groups),
    ('NEED POSITION FIRST', need_position),
]:
    print(f"\n{label}: {len(sessions)}")
    for s in sessions:
        print(f"    {s!r},")

n_done = int((candidates['next_step'] == 'done').sum())
n_no_ephys = int((candidates['next_step'] == 'no ephys (cannot sort)').sum())
print(f"\n{n_done} session(s) fully done; {n_no_ephys} have no ephys (behavior/photometry only).")


READY TO SORT (run the full pipeline): 1
    'IM-1947_20260404_.nwb',

READY TO DECODE (sorted already): 0

NEED SORT GROUPS FIRST: 0

NEED POSITION FIRST: 0

26 session(s) fully done; 24 have no ephys (behavior/photometry only).
